# NSD: HRF stability and beta-series changes across sessions

This notebook combines **sub-07, sessions 10–19**. It asks:

1. How similar are the selected **full HRF curves at the same grayordinate** across sessions?
2. How does HRF optimization change mean trial beta, descriptive task t,
   task-added BOLD ΔR², and within-run beta–RT associations at that grayordinate?

Completed session results are reused. **Missing sessions are fitted automatically**
by running `nsd_workflow.ipynb` in a separate kernel for each session. With the
513-HRF library and fractional CV, this can be a long computation. Each completed
session is saved before the next starts; reruns resume from those outputs.
Set `fit_missing=False` for a strictly read-only session-loading pass.


## 1. Configuration

Set `NSD_BIDS_ROOT` in the environment, or define
`NSD_MULTI_CONFIG = {"bids_root": "/path/to/NSD/BIDS"}` before this cell.
`output_root` locates the existing single-session derivatives and receives aggregate
outputs under `sub-07/func/`. Change `sessions` to select a different set.

`analysis_config` forwards optional fitting settings (for example `n_jobs`,
`block_size`, `hrf_n_samples`, `ridge_mode`, or `max_grayordinates`). Scientific
settings inherit from the first existing session; without one, the single-session
notebook defaults apply. Conflicting settings stop before fitting.
Default comparisons are OLS and fractional CV; use `estimators=["OLS"]` for OLS only.


In [ ]:
from pathlib import Path
import sys

repo = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "src/boldtailor").is_dir())
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from examples.NSD.notebook_paths import notebook_paths
from examples.NSD.multisession_workflow import ensure_session_outputs
from examples.NSD.multisession_inputs import load_sessions
from examples.NSD.multisession_analysis import analyze_sessions
from examples.NSD.multisession_outputs import beta_summary_table, save_multisession
from examples.NSD.multisession_plots import (
    METRIC_LABELS, beta_change_figure, comparison_surface, choose_grayordinate,
    grayordinate_hrf_figure, grayordinate_beta_figure, glm_effect_figures,
    peak_time_surface,
)
from examples.NSD.session_hrf_plots import agreement_figure
from boldtailor.workflow.surfaces import find_surface_meshes

config = dict(subject="sub-07", sessions=[f"ses-nsd{i}" for i in range(10, 20)],
              estimators=["OLS", "FractionalCV"], fit_missing=True,
              analysis_config={}, surface_maps=True, surface_meshes=None,
              grayordinate=None, save_results=True)
config.update(globals().get("NSD_MULTI_CONFIG", {}))
config.update(notebook_paths(config))
output = Path(config["output_root"])
figures = {}


## 2. Prepare or reuse each session

Sessions run sequentially; fitting uses the single-session workflow's workers.
Existing complete results are not refitted. Missing or incomplete session results
are regenerated with transactional replacement of that session's notebook outputs.
Summary loading uses compact maps, not all sessions' trial-beta arrays at once.


In [ ]:
fit_config = {**config["analysis_config"],
              **{key: config[key] for key in ("bids_root", "fmriprep_root", "output_root", "subject")}}
status = ensure_session_outputs(fit_config, config["sessions"],
                                estimators=config["estimators"], fit_missing=config["fit_missing"])
display(status)
loaded = load_sessions(output, config["subject"], config["sessions"],
                       estimators=config["estimators"])
result = analyze_sessions(loaded)
brain = loaded["brain"]
meshes = (find_surface_meshes(config["fmriprep_root"], config["subject"],
                             paths=config["surface_meshes"]) if config["surface_maps"] else None)
print(f"{len(loaded['sessions'])} sessions; {len(brain):,} grayordinates")


## 3. Full-HRF consistency across sessions

For **each grayordinate**, correlate the full selected HRF curves across time for
all unordered session pairs. Curves use the same stored time grid, including tails;
there is no peak alignment. Correlation measures shape agreement, not amplitude.
All sessions must have the same exact HRF library and spatial axis.

The baseline is each session's correlation with the canonical HRF. Similarity to
that shared baseline can explain high between-session correlations. The plots show
matched pairwise comparisons and their difference from this baseline. Session pairs
share observations: these are descriptive summaries, not independent replicates.

The cortical map shows **Δr = between-session r − matched canonical baseline r**,
averaged over valid session pairs at each grayordinate. Each pair's baseline is the
mean of its two session-to-canonical correlations. The diverging scale is centered
on zero: positive values indicate greater cross-session shape agreement than the
canonical baseline, while negative values indicate less. This measures HRF shape
consistency, not improvement in BOLD prediction.


In [ ]:
hrf = result["hrf"]
figures["HRFAgreement"] = agreement_figure(hrf)
display(figures["HRFAgreement"])
plt.close(figures["HRFAgreement"])
display(pd.DataFrame({"pair": hrf["pair_names"],
                      "valid_grayordinates": np.isfinite(hrf["pairwise"]).sum(axis=1)}))
if meshes is not None:
    figures["HRFSurface"] = comparison_surface(
        {"HRF Δr": hrf["summary"][2]},
        brain, meshes, label="Δr: between-session − canonical baseline",
        title="Full-HRF shape agreement above canonical baseline")
    figures["HRFSurface"].axes[-1].yaxis.set_label_position("left")
    display(figures["HRFSurface"])


## 3a. Mean HRF time-to-peak across sessions

At each grayordinate, average the **time-to-peak of the selected HRF in each
session**, in seconds. Peak time is the time of the maximum of the full stored
HRF curve on the library's 0.1-second grid, including any onset delay. It is not
the response-delay parameter or the peak time of an averaged HRF curve.

Valid sessions receive equal weight, including sessions selecting the canonical
HRF. Missing selections are omitted; a location with one valid session shows
that session's peak time, and locations with no estimates remain gray. The
sequential color scale covers finite cortical values. The saved
`desc-multisessionHRF_stat-peaktime.dscalar.nii` contains mean peak time and
contributing-session count for every grayordinate.


In [ ]:
if meshes is not None:
    figures["HRFPeakTime"] = peak_time_surface(result, brain, meshes)
    display(figures["HRFPeakTime"])
else:
    print("Cortical surfaces unavailable; mean peak times and counts will still be saved.")


## 3b. Mean task response and RT effect across sessions

These conventional GLM coefficient maps show the **mean task response** and
**mean RT slope**, separately from the single-trial beta summaries below.
Each figure shows canonical-HRF, optimized-HRF, and **optimized − canonical**
maps on a shared, zero-centered color scale. Task coefficients use native signal
units; RT coefficients use native signal units per second of response time.

At each grayordinate, each effect uses the same sessions with finite values for
both models, weighted equally, and requires at least two matched sessions.
The delta is the mean within-session coefficient difference. Missing locations
remain gray. The task coefficient is the response on trial_type 0 trials at the
run-mean observed RT, with the missing-RT indicator zero in runs that include it.

These are unthresholded effect estimates, not significance maps. Positive deltas
mean a more positive coefficient, not necessarily a stronger effect or better
prediction. Native signal scaling can differ between sessions. Maps and valid
session counts are saved as `desc-multisessionGLM_stat-...` CIFTIs.


In [ ]:
if meshes is not None:
    glm_figures = glm_effect_figures(result, brain, meshes)
    figures.update(glm_figures)
    for figure in glm_figures.values():
        display(figure)
else:
    print("Cortical surfaces unavailable; GLM coefficient summaries will still be saved.")


## 4. Paired changes in task and RT results

Within each session and grayordinate, subtract **canonical from optimized**:

| Metric | Interpretation of a positive change |
| --- | --- |
| Mean trial beta | Higher signed mean beta in native signal units |
| Descriptive task t | Higher signed mean-beta t statistic; the difference is not a t test |
| Task-added ΔR² | More BOLD variance attributed to trial regressors beyond confounds |
| Signed beta–RT r | A more positive association; not necessarily a stronger association |
| Absolute beta–RT r | A stronger association in magnitude, regardless of sign |

Means give sessions equal weight, using the same finite session pairs for both
models. At least **two valid sessions** are needed at each grayordinate. Counts,
SDs of paired changes, and the fraction of sessions with a positive change are
exported. Native beta units can differ in scale across sessions; standardized t,
ΔR², and RT correlations provide complementary views.

OLS isolates the HRF change from ridge regularization. Fractional-CV models select
fractions separately for canonical and optimized HRFs, so their differences include
both choices. RT participates in CV tuning, and HRFs use the fitted session's data:
these are **descriptive in-sample changes**, not evidence of out-of-sample improvement.


In [ ]:
display(beta_summary_table(result))
figures["BetaChanges"] = beta_change_figure(result)
display(figures["BetaChanges"])

if meshes is not None:
    for i, metric in enumerate(result["metrics"]):
        maps = {e: values["difference_mean"][i] for e, values in result["beta"].items()}
        name = "Change_" + metric
        figures[name] = comparison_surface(
            maps, brain, meshes, label="Optimized − canonical: " + METRIC_LABELS[metric],
            title="Mean paired session change: " + METRIC_LABELS[metric], t_scale=metric == "task_t")
        display(figures[name])


## 5. Inspect a grayordinate

Set `grayordinate` in the configuration to a **zero-based CIFTI column index**.
If omitted, choose the first cortical grayordinate with at least two HRF selections
(falling back to any valid grayordinate). This choice does not rank optimization gains.
Change `config["grayordinate"]` and re-run this section to inspect another location.


In [ ]:
vertex = choose_grayordinate(loaded, result, config["grayordinate"])
if vertex is None:
    print("No grayordinate has HRF estimates in two sessions.")
else:
    print(f"Grayordinate {vertex}: {brain.name[vertex]}, surface vertex {brain.vertex[vertex]}")
    figures["GrayordinateHRFs"] = grayordinate_hrf_figure(loaded, result, vertex)
    display(figures["GrayordinateHRFs"])
    figures["GrayordinateBetas"], detail = grayordinate_beta_figure(loaded, vertex)
    display(figures["GrayordinateBetas"])
    display(detail)


## 6. Save combined results

Aggregate CIFTIs preserve all grayordinate positions, including NaNs and contribution
counts. Files include every HRF session-pair correlation, paired beta summaries,
per-session changes, GLM task/RT coefficient means and deltas, tables, figures,
and metadata with source-file hashes.
Only `desc-multisession...` aggregate outputs are replaced on rerun; the contributing
single-session outputs are preserved. Set `save_results=False` to skip publication.


In [ ]:
if config["save_results"]:
    paths = save_multisession(output, loaded, result, figures=figures)
    print(f"Saved {len(paths)} aggregate files under {output / config['subject'] / 'func'}")
else:
    paths = ()
    print("Aggregate saving disabled.")
